#### Question 1

 docker exec -it workshop-redpanda-1 rpk version
rpk version: v25.3.9

#### Question 2

```
docker exec -it workshop-redpanda-1 rpk topic create green-trips
```

In [ ]:
import pandas as pd
columns = ["PULocationID", "DOLocationID", "trip_distance", "total_amount", "lpep_pickup_datetime"]
df = pd.read_parquet("workshop/green_tripdata_2025-10.parquet", columns=columns).head(1000)


,PULocationID,DOLocationID,trip_distance,total_amount,lpep_pickup_datetime
0,247,69,0.70,10.00,2025-10-01 00:21:47
1,66,25,1.61,16.68,2025-10-01 00:14:03
2,244,244,0.00,13.20,2025-10-01 00:16:44
3,95,170,10.37,67.85,2025-10-01 00:07:36
4,82,138,4.07,34.12,2025-09-30 21:10:29
...,...,...,...,...,...
995,43,151,1.53,14.70,2025-10-01 17:08:02
996,166,116,1.90,24.57,2025-10-01 17:22:39
997,41,41,0.83,10.50,2025-10-01 17:53:05
998,82,138,3.68,28.70,2025-10-01 17:06:11


In [14]:
import pandas as pd
import json
import math
from kafka import KafkaProducer
from time import time

# Load only the required columns
columns = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "tip_amount",
    "total_amount",
]
df = pd.read_parquet("workshop/green_tripdata_2025-10.parquet", columns=columns)

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

t0 = time()

for _, row in df.iterrows():
    record = row.to_dict()
    for key, value in record.items():
        if hasattr(value, "isoformat"):
            record[key] = value.isoformat()
        elif isinstance(value, float) and math.isnan(value):
            record[key] = None  # replace NaN with null

    producer.send("green-trips", value=record)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 6.65 seconds


#### Question 3. Consumer - trip distance

In [13]:
import json
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    "green-trips",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=5000,  # stop after 5s of no new messages
)

count = 0
for msg in consumer:
    if msg.value.get("trip_distance", 0) > 5.0:
        count += 1

print(f"Trips with trip_distance > 5.0: {count}")

Trips with trip_distance > 5.0: 8506


#### Question 4

```sql
SELECT window_start, PULocationID, num_trips FROM green_trips_5min ORDER BY num_trips DESC LIMIT 3;
```